In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [5]:
%%writefile src/stability.py
import numpy as np
import pandas as pd

from src import config, model, rules, splits
from src.metrics import pythag

def team_strengths(teams):
    t=teams.set_index('team')
    a,b=config.TS_TEAMS
    return t.loc[a,'pythag'] , t.loc[b,'pythag']


Writing src/stability.py


In [11]:
%%writefile -a src/stability.py
# ---------- 實驗 1：模擬次數與亂數 ----------
def exp_seeds(teams, n_seeds=30, n=10_000):
    pa, pb = team_strengths(teams)
    probs = [model.simulate_series(pa, pb, config.TS_HOME_PATTERN, config.TS_NEED,
                                   config.HOME_ADVANTAGE, n=n, seed=s)[0] for s in range(n_seeds)]
    return pd.Series(probs, name="奪冠機率")

Appending to src/stability.py


In [13]:

import pandas as pd
from src import stability as st
s = st.exp_seeds(pd.read_csv('data/clean/teams.csv'))
print(f'平均 {s.mean():.1%}、標準差 {s.std():.2%}、最小 {s.min():.1%}、最大 {s.max():.1%}')


平均 67.1%、標準差 0.46%、最小 65.8%、最大 67.8%


In [16]:
%%writefile -a src/stability.py
# ---------- 實驗 2：參數敏感度 ----------
def exp_sensitivity(teams):
    """一次只改一個參數，看系列賽奪冠機率和中立場地單場勝率怎麼變。"""
    t = teams.set_index("team")
    a, b = config.TS_TEAMS
    rows = []
    settings = [("主場優勢", v, dict(hfa=v)) for v in [0, 0.015, 0.03, 0.045, 0.06]] + \
               [("畢氏指數", v, dict(exp=v)) for v in [1.8, 1.83, 1.9, 2.0]]
    for name, value, kw in settings:
        exp = kw.get("exp", 1.83)
        hfa = kw.get("hfa", config.HOME_ADVANTAGE)
        pa = pythag(t.loc[a, "RS"], t.loc[a, "RA"], exp)
        pb = pythag(t.loc[b, "RS"], t.loc[b, "RA"], exp)
        series = model.simulate_series(pa, pb, config.TS_HOME_PATTERN, config.TS_NEED, hfa)[0]
        rows.append({"參數": name, "數值": value, "單場勝率（中立）": model.log5(pa, pb), "奪冠機率": series})
    return pd.DataFrame(rows)

Appending to src/stability.py


In [17]:

import pandas as pd
from src import stability as st
print(st.exp_sensitivity(pd.read_csv('data/clean/teams.csv')).round(3).to_string())


     參數     數值  單場勝率（中立）   奪冠機率
0  主場優勢  0.000     0.570  0.657
1  主場優勢  0.015     0.570  0.668
2  主場優勢  0.030     0.570  0.681
3  主場優勢  0.045     0.570  0.696
4  主場優勢  0.060     0.570  0.710
5  畢氏指數  1.800     0.569  0.674
6  畢氏指數  1.830     0.570  0.676
7  畢氏指數  1.900     0.573  0.681
8  畢氏指數  2.000     0.577  0.686


In [20]:
%%writefile -a src/stability.py
def default_game_prob(teams, batters, pitchers, lg, sp_innings=6, a_home=True):
    """用「兩隊投球局數最多的先發＋打席最多的 9 人」算單場勝率，給實驗 2 測先發局數用。"""
    t = teams.set_index("team")
    a, b = config.TS_TEAMS
    ctx = model.league_context(teams, pitchers, lg["wOBA"])
    tp = pd.DataFrame(model.team_pitching(pitchers)).set_index("abbr")
    side = {}
    for team in (a, b):
        abbr = t.loc[team, "abbr"]
        sp = pitchers[(pitchers["abbr"] == abbr) & (pitchers["role"] == "先發")].nlargest(1, "IP").iloc[0]
        lu = batters[batters["abbr"] == abbr].nlargest(9, "PA")
        side[team] = (abbr, sp, lu)
    (aa, sa, la), (ab, sb, lb) = side[a], side[b]
    w = lambda lu, sp: lu.apply(lambda r: rules.bat_woba(r, sp["throws"]), axis=1).mean()
    ra = model.expected_runs(w(la, sb), la["wOBA_calc"].mean(), sb["FIP_calc"], tp.loc[ab, "bullpen_fip"],
                             tp.loc[ab, "DER"], ctx, sp_innings, a_home)
    rb = model.expected_runs(w(lb, sa), lb["wOBA_calc"].mean(), sa["FIP_calc"], tp.loc[aa, "bullpen_fip"],
                             tp.loc[aa, "DER"], ctx, sp_innings, not a_home)
    return model.win_prob_from_runs(ra, rb)

Appending to src/stability.py


In [21]:

from src import stability as st
from src.data import load
teams, batters, pitchers, const, lg = load()
for i in (5, 6, 7):
    print(i, '局：', round(st.default_game_prob(teams, batters, pitchers, lg, sp_innings=i), 3))


5 局： 0.506
6 局： 0.5
7 局： 0.494


In [22]:

!git add .
!git commit -m "Day 29: 實驗 1、2"
!git push

[main 67995fc] Day 29: 實驗 1、2
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 5 files changed, 391 insertions(+), 3 deletions(-)
 create mode 100644 D14.ipynb
 create mode 100644 src/stability.py
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 8 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 5.49 KiB | 5.49 MiB/s, done.
Total 9 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (6/6), completed with 6 local objects.
To https://github.com/le

In [24]:
from src import fetch

games = fetch.fetch_games()
print(len(games))   # 應該剛好是 360（例行賽場數）

360


In [25]:
%%writefile -a src/stability.py
# ---------- 實驗 3：資料重抽（Bootstrap） ----------
def exp_bootstrap_series(teams, games, n_boot=200, seed=0):
    """從例行賽 360 場比賽中「有放回地」重抽，重算得失分 → 畢氏勝率 → 奪冠機率。"""
    rng = np.random.default_rng(seed)
    a, b = config.TS_TEAMS
    out = []
    for _ in range(n_boot):
        g = games.iloc[rng.integers(0, len(games), len(games))]
        rs_ra = {}
        for team in (a, b):
            home, away = g[g["home"] == team], g[g["away"] == team]
            rs_ra[team] = (home["home_runs"].sum() + away["away_runs"].sum(),
                           home["away_runs"].sum() + away["home_runs"].sum())
        pa, pb = pythag(*rs_ra[a]), pythag(*rs_ra[b])
        out.append(model.simulate_series(pa, pb, config.TS_HOME_PATTERN, config.TS_NEED,
                                         config.HOME_ADVANTAGE, n=2_000, seed=1)[0])
    return pd.Series(out, name="奪冠機率")

Appending to src/stability.py


In [26]:

import pandas as pd
from src import fetch, stability as st
b = st.exp_bootstrap_series(pd.read_csv('data/clean/teams.csv'), fetch.fetch_games())
print(b.quantile([0.025, 0.5, 0.975]).round(3))


0.025    0.345
0.500    0.673
0.975    0.904
Name: 奪冠機率, dtype: float64


In [27]:
%%writefile -a src/stability.py
def exp_bootstrap_rules(pa, scale, sp, lineup, lg, n_boot=200, seed=0):
    """重抽逐打席資料，重算這位先發的拆分數據，統計每條弱點規則有幾成的機率成立。"""
    rng = np.random.default_rng(seed)
    base = splits.prepare(pa, scale)
    mine = base[base["pitcher_id"] == sp["player_id"]]
    counts = {}
    for _ in range(n_boot):
        s = mine.iloc[rng.integers(0, len(mine), len(mine))]
        p = sp.copy()
        for hand in ("L", "R"):
            sub = s[s["b_hand"] == hand]
            p[f"PA_vs{hand}"], p[f"wOBA_vs{hand}"] = len(sub), sub["woba_val"].mean()
        p["wOBA_round3"] = s.loc[s["pa_round"] == 3, "woba_val"].mean()
        for sentence in rules.pitcher_weak(p, lineup, lg):
            key = sentence.split("：")[0]          # 用冒號前的標題當作規則名稱
            counts[key] = counts.get(key, 0) + 1
    return pd.Series(counts).div(n_boot).sort_values(ascending=False).rename("成立比例")

Appending to src/stability.py


In [28]:

import pandas as pd
from src import stability as st
from src.data import load
teams, batters, pitchers, const, lg = load()
pa = pd.read_csv('data/clean/pa.csv', dtype=str, keep_default_na=False)
for abbr, opp in [('象', '龍'), ('龍', '象')]:
    sp = pitchers[(pitchers['abbr'] == abbr) & (pitchers['role'] == '先發')].nlargest(1, 'IP').iloc[0]
    lineup = batters[batters['abbr'] == opp].nlargest(9, 'PA')
    print(sp['name'], st.exp_bootstrap_rules(pa, const['woba_scale'], sp, lineup, lg).round(2).to_dict())


勝騎士 {'每九局被 0.76 支全壘打，容易挨長打。': 1.0, '對左打明顯較弱': 0.36, '對右打明顯較弱': 0.16, '第三輪打線開始被看穿': 0.02}
鋼龍 {'對右打明顯較弱': 0.58, '第三輪打線開始被看穿': 0.44, '沒有明顯弱點': 0.25, '對左打明顯較弱': 0.04}


In [29]:
%%writefile -a src/stability.py
def rule_stability_table(pa, scale, teams, batters, pitchers, lg, n_boot=200):
    """台灣大賽兩隊每位先發的每條弱點規則，在重抽中成立的比例。存起來給預測頁過濾用。"""
    t = teams.set_index("team")
    a, b = config.TS_TEAMS
    rows = []
    for team, opp in ((a, b), (b, a)):
        abbr, opp_abbr = t.loc[team, "abbr"], t.loc[opp, "abbr"]
        lineup = batters[batters["abbr"] == opp_abbr].nlargest(9, "PA")
        for _, sp in pitchers[(pitchers["abbr"] == abbr) & (pitchers["role"] == "先發")].iterrows():
            ratio = exp_bootstrap_rules(pa, scale, sp, lineup, lg, n_boot)
            rows += [{"pitcher_id": sp["player_id"], "rule": k, "ratio": v} for k, v in ratio.items()]
    return pd.DataFrame(rows)

Appending to src/stability.py


In [30]:

import pandas as pd
from src import stability as st
from src.data import load
teams, batters, pitchers, const, lg = load()
pa = pd.read_csv('data/clean/pa.csv', dtype=str, keep_default_na=False)
st.rule_stability_table(pa, const['woba_scale'], teams, batters, pitchers, lg).to_csv('data/clean/rule_stability.csv', index=False)

In [31]:
%%writefile -a src/rules.py
# 規則名稱 → 這條規則用到的拆分數據欄位
RULE_COLUMNS = {"對左打明顯較弱": ["wOBA_vsL", "wOBA_vsR"], "對右打明顯較弱": ["wOBA_vsL", "wOBA_vsR"],
                "第三輪打線開始被看穿": ["wOBA_round3"]}

def stable_pitcher(sp, opp_lineup, lg, stab, threshold=0.8):
    """
    Day 30：把「在 Bootstrap 重抽中成立不到 threshold」的弱點所用到的拆分數據清空，
    之後的弱點分析和應對策略就都不會用到不穩定的結論。回傳（清理後的投手, 被隱藏幾項）。
    """
    if stab is None:
        return sp, 0
    ratio = stab[stab["pitcher_id"] == sp["player_id"]].set_index("rule")["ratio"]
    sp = sp.copy()
    hidden = 0
    for sentence in pitcher_weak(sp, opp_lineup, lg):
        rule = sentence.split("：")[0]
        if rule in RULE_COLUMNS and ratio.get(rule, 1.0) < threshold:
            for col in RULE_COLUMNS[rule]:
                sp[col] = float("nan")
            hidden += 1
    return sp, hidden

Appending to src/rules.py


In [32]:
%%writefile -a src/data.py
@st.cache_data
def load_rule_stability():
    """每條弱點規則在重抽中成立的比例（Day 30）；還沒跑就回傳 None。"""
    path = CLEAN / "rule_stability.csv"
    return pd.read_csv(path) if path.exists() else None

Appending to src/data.py


In [33]:
!git add .
!git commit -m "Day 30: 實驗 3 Bootstrap"
!git push

[main 7a2b759] Day 30: 實驗 3 Bootstrap
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 6 files changed, 456 insertions(+), 20 deletions(-)
 create mode 100644 data/clean/rule_stability.csv
Enumerating objects: 22, done.
Counting objects: 100% (22/22), done.
Delta compression using up to 8 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (12/12), 9.39 KiB | 9.39 MiB/s, done.
Total 12 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (8/8), completed with 8 local objects.
To https://github.com/leo85

In [34]:
%%writefile -a src/stability.py
# ---------- 實驗 4：資料截止日 ----------
def exp_cutoff(games, cutoffs=("2026-08-31", "2026-09-15", "2026-10-06")):
    a, b = config.TS_TEAMS
    rows = []
    for cut in cutoffs:
        g = games[games["date"] <= cut]
        rs_ra = {}
        for team in (a, b):
            home, away = g[g["home"] == team], g[g["away"] == team]
            rs_ra[team] = (home["home_runs"].sum() + away["away_runs"].sum(),
                           home["away_runs"].sum() + away["home_runs"].sum())
        pa, pb = pythag(*rs_ra[a]), pythag(*rs_ra[b])
        p = model.simulate_series(pa, pb, config.TS_HOME_PATTERN, config.TS_NEED, config.HOME_ADVANTAGE)[0]
        rows.append({"截止日": cut, "場數": len(g), f"{a} 畢氏勝率": pa, f"{b} 畢氏勝率": pb, "奪冠機率": p})
    return pd.DataFrame(rows)

Appending to src/stability.py


In [35]:

from src import fetch, stability as st
print(st.exp_cutoff(fetch.fetch_games()).round(3).to_string())


          截止日   場數  味全龍 畢氏勝率  中信兄弟 畢氏勝率   奪冠機率
0  2026-08-31  282     0.627      0.457  0.841
1  2026-09-15  314     0.591      0.480  0.753
2  2026-10-06  360     0.578      0.508  0.676


In [36]:
%%writefile scripts/run_experiments.py
"""上線後實驗：一次跑完實驗 1–4，印出報告並存成 data/clean/stability_summary.json。"""
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
import pandas as pd

from src import config, fetch, stability as st
from src.data import load_raw

teams, batters, pitchers, const, lg = load_raw()
games = fetch.fetch_games()
pa = pd.read_csv(ROOT / "data/clean/pa.csv", dtype=str, keep_default_na=False)
a, b = config.TS_TEAMS
result = {}

# 實驗 1
s = st.exp_seeds(teams)
result["exp1"] = {"平均": s.mean(), "標準差": s.std(), "通過": bool(s.std() < 0.01)}

# 實驗 2
sens = st.exp_sensitivity(teams)
innings = {i: st.default_game_prob(teams, batters, pitchers, lg, sp_innings=i) for i in (5, 6, 7)}
spread = max(sens.groupby("參數")["奪冠機率"].agg(lambda x: x.max() - x.min()).max(),
             max(innings.values()) - min(innings.values()))
result["exp2"] = {"最大變動": spread, "先發局數": innings,
                  "看好的球隊不變": bool((sens["奪冠機率"] > 0.5).all() or (sens["奪冠機率"] < 0.5).all()),
                  "通過": bool(max(innings.values()) - min(innings.values()) <= 0.05)}

# 實驗 3
boot = st.exp_bootstrap_series(teams, games)
lo, hi = boot.quantile([0.025, 0.975])
result["exp3"] = {"奪冠機率95%區間": [lo, hi], "區間寬度": hi - lo, "通過": bool(hi - lo <= 0.10)}
stab = st.rule_stability_table(pa, const["woba_scale"], teams, batters, pitchers, lg)
stab.to_csv(ROOT / "data/clean/rule_stability.csv", index=False)
result["exp3"]["規則穩定比例"] = float((stab["ratio"] >= 0.8).mean())

# 實驗 4
cut = st.exp_cutoff(games)
drift = cut["奪冠機率"].max() - cut["奪冠機率"].min()
result["exp4"] = {"各截止日": cut["奪冠機率"].round(3).tolist(), "最大差距": drift, "通過": bool(drift <= 0.10)}

out = ROOT / "data/clean/stability_summary.json"
out.write_text(json.dumps(result, ensure_ascii=False, indent=1, default=float), encoding="utf-8")
for k, v in result.items():
    print(k, json.dumps(v, ensure_ascii=False, default=lambda x: round(float(x), 3)))

Writing scripts/run_experiments.py


In [37]:
!python scripts/run_experiments.py

2026-11-05 20:27:09.870 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:27:09.870 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:27:09.871 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-05 20:27:09.871 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
exp1 {"平均": 0.67072, "標準差": 0.004571380233734595, "通過": true}
exp2 {"最大變動": 0.053099999999999925, "先發局數": {"5": 0.5057455641084038, "6": 0.4998950628748998, "7": 0.49373908457796556}, "看好的球隊不變": true, "通過": true}
exp3 {"奪冠機率95%區間": [0.345375, 0.90375], "區間寬度": 0.5583750000000001, "通過": false, "規則穩定比例": 0.34328358208955223}
exp4 {"各截止日": [0.841, 0.753, 0.676], "最大差距": 0.16510000000000002, "通過": false}


In [38]:
%%writefile -a src/data.py
@st.cache_data
def load_stability():
    """上線後實驗的摘要（Day 31）；還沒跑實驗就回傳 None。"""
    path = CLEAN / "stability_summary.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

Appending to src/data.py
